In [ ]:
!pip install transformers sentence-transformers gradio --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 55.7 MB/s eta 0:00:00


In [ ]:

from google.colab import files
import pandas as pd
import io

uploaded = files.upload()


for file_name in uploaded.keys():
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))
    print(f" Successfully loaded: {file_name}")
    display(df.head())


Saving Harsh college_info_dataset_input_output.csv to Harsh college_info_dataset_input_output.csv
 Successfully loaded: Harsh college_info_dataset_input_output.csv


,input,output
0,What is the admission process?,The admission process includes online applicat...
1,What entrance exams are accepted?,"The college accepts JEE Main, COMEDK, and stat..."
2,What are the eligibility criteria?,"For B.Tech, candidates must have passed 10+2 w..."
3,Which documents are required for admission?,"You need 10th and 12th marksheets, transfer ce..."
4,What are the important dates?,"Application starts in March, entrance exams ar..."


In [ ]:


df.columns = df.columns.str.strip().str.lower()
print("Columns in dataset:", df.columns.tolist())


if 'input' in df.columns and 'output' in df.columns:
    print(" Sample input:", df['input'].iloc[0])
    print(" Sample output:", df['output'].iloc[0])
elif 'question' in df.columns and 'answer' in df.columns:

    print(" Sample question:", df['question'].iloc[0])
    print(" Sample answer:", df['answer'].iloc[0])
else:
    print(" Could not find 'input'/'output' or 'question'/'answer' columns in the dataset.")

Columns in dataset: ['input', 'output']
 Sample input: What is the admission process?
 Sample output: The admission process includes online application submission, appearing for the entrance exam (like JEE, CET), and participating in counseling and document verification.


In [ ]:
from sentence_transformers import SentenceTransformer, util

model_embed = SentenceTransformer('all-MiniLM-L6-v2')
faq_embeddings = model_embed.encode(df['input'].tolist(), convert_to_tensor=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from transformers import pipeline

gen_pipeline = pipeline("text2text-generation", model="google/flan-t5-base")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
def get_best_match(user_input):

    global df, faq_embeddings

    print("User input received:", user_input)
    print("First few questions:", df['input'].tolist()[:3])


    user_embedding = model_embed.encode(user_input, convert_to_tensor=True)


    scores = util.cos_sim(user_embedding, faq_embeddings)
    best_idx = int(scores.argmax())

    print("Best match index:", best_idx)
    print("Matched question:", df.iloc[best_idx]['input'])
    print("Matched answer:", df.iloc[best_idx]['output'])

    return df.iloc[best_idx]['output']

def chatbot_response(user_input):
    global df
    try:
        matched_answer = get_best_match(user_input)
        prompt = f"Answer like a helpful college counselor: {matched_answer}"
        result = gen_pipeline(prompt, max_new_tokens=100)[0]['generated_text']
        return result
    except Exception as e:
        print(" Error in chatbot_response:", str(e))
        return "Error: Something went wrong. Please check the console logs."


In [ ]:

print("Manual Test:")
user_q = "How do I apply for admission?"
response = chatbot_response(user_q)
print("Bot Response:", response)


Manual Test:
User input received: How do I apply for admission?
First few questions: ['What is the admission process?', 'What entrance exams are accepted?', 'What are the eligibility criteria?']
Best match index: 0
Matched question: What is the admission process?
Matched answer: The admission process includes online application submission, appearing for the entrance exam (like JEE, CET), and participating in counseling and document verification.
Bot Response: The admissions process includes online application submission, appearing for the entrance exam (like JEE, CET), and participating in counseling and document verification.


In [ ]:
import gradio as gr

THRESHOLD = 0.6  # Cosine similarity threshold

def chatbot_gradio(user_input):
    try:
        # Encode user input
        user_embedding = model_embed.encode(user_input, convert_to_tensor=True)
        scores = util.cos_sim(user_embedding, faq_embeddings)
        best_idx = int(scores.argmax())
        best_score = float(scores[0][best_idx])

        # FAQ matched answer
        matched_answer = df.iloc[best_idx]['output']

        # Use matching if score is good enough
        if best_score >= THRESHOLD:
            response = matched_answer
            source = "✅ Answer from Matched FAQ"
        else:
            # Generate using Flan-T5
            prompt = f"Answer the following college-related question like a helpful counselor: {user_input}"
            response = gen_pipeline(prompt, max_new_tokens=100)[0]['generated_text']
            source = "🤖 Answer from Flan-T5 Generator"

        # Format response
        return f"{source}\n\n{response.strip()} (Similarity Score: {best_score:.2f})"

    except Exception as e:
        return f"❌ Error: {str(e)}"

# Launch Gradio interface
gr.Interface(
    fn=chatbot_gradio,
    inputs=gr.Textbox(lines=2, placeholder="Ask a college-related question here..."),
    outputs=gr.Textbox(label="Chatbot Response"),
    title="🎓 College Inquiry Chatbot",
    description="Ask your question. The chatbot will answer using either matched FAQ or Flan-T5 model based on similarity."
).launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e35bed32ea62bdff56.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
